# RunLoadTest - Long Iteration Benchmark

Optimized for 100+ iteration runs with minimal output.
Date-shifted queries bypass AAS cache per iteration.

In [ ]:
# Imports
import copy, glob, json, os, re, time, sys
import numpy as np
import pandas as pd
import polars as pl
import sempy.fabric as fabric
import notebookutils
import ipywidgets as widgets
from IPython.display import display
from datetime import datetime, date, timezone, timedelta as td
from datetime import datetime as dt

sys.path.insert(0, '/lakehouse/default/Files/test/PerfScenarios/lib')
from applicationinsights import TelemetryClient
from FabricLoadTestTelemetry import send_loadtest_telemetry

VERBOSE = False

In [ ]:
# Configuration Widgets

pst_tz = timezone(td(hours=-8))
today_pst = datetime.now(pst_tz).date()

test_label_input = widgets.Text(
    value='', description='Test Label:',
    placeholder='e.g., Baseline, Enhancement',
    style={'description_width': 'initial'}
)
environment_dropdown = widgets.Dropdown(
    options=[('Test', 'Payments @ Microsoft [Test]'),
             ('Prod', 'Payments @ Microsoft [Prod]')],
    value='Payments @ Microsoft [Test]',
    description='Environment:',
    style={'description_width': 'initial'}
)
concurrent_users_dropdown = widgets.Dropdown(
    options=[1, 2, 3, 5, 10], value=1,
    description='Sessions:',
    style={'description_width': 'initial'}
)
execution_mode_dropdown = widgets.Dropdown(
    options=[('Parallel', False), ('Sequential', True)],
    value=True, description='Query Mode:',
    style={'description_width': 'initial'}
)
max_parallel_dropdown = widgets.Dropdown(
    options=[5, 10, 15, 20, 25, 50, 100], value=10,
    description='Max Parallel:',
    style={'description_width': 'initial'}
)
query_start_date = widgets.DatePicker(
    description='Start Date:', value=date(2023, 10, 1),
    style={'description_width': 'initial'}
)
query_end_date = widgets.DatePicker(
    description='End Date:', value=today_pst,
    style={'description_width': 'initial'}
)
period_type_dropdown = widgets.Dropdown(
    options=['Daily', 'Weekly', 'Monthly', 'Quarterly', 'Yearly'],
    value='Weekly', description='Period Type:',
    style={'description_width': 'initial'}
)
iterations_dropdown = widgets.Dropdown(
    options=[1, 2, 3, 5, 10, 15, 20, 25, 50, 100, 150, 200],
    value=100, description='Iterations:',
    style={'description_width': 'initial'}
)
clear_aas_cache_checkbox = widgets.Checkbox(
    value=True, description='Clear AAS Cache per iteration',
    style={'description_width': 'initial'}
)
refresh_fabric_checkbox = widgets.Checkbox(
    value=True, description='Trigger Fabric Refresh (async)',
    style={'description_width': 'initial'}
)

print('Load Test Configuration')
print('-' * 40)
for w in [test_label_input, environment_dropdown, concurrent_users_dropdown,
          execution_mode_dropdown, max_parallel_dropdown, iterations_dropdown,
          query_start_date, query_end_date, period_type_dropdown,
          clear_aas_cache_checkbox, refresh_fabric_checkbox]:
    display(w)

In [ ]:
# Helper Functions

def shift_dates_in_runtime_json(runtime_json: dict, day_offset: int) -> dict:
    """Shift all DAX query dates by day_offset to bypass AAS cache."""
    shifted = copy.deepcopy(runtime_json)
    START_P = r">=\s*DATE\s*\(\s*(\d{4})\s*,\s*(\d{1,2})\s*,\s*(\d{1,2})\s*\)"
    END_P = r"<\s*DATE\s*\(\s*(\d{4})\s*,\s*(\d{1,2})\s*,\s*(\d{1,2})\s*\)"

    def _shift(match, op):
        y, m, d = int(match.group(1)), int(match.group(2)), int(match.group(3))
        sd = datetime(y, m, d) + td(days=day_offset)
        return f"{op} DATE({sd.year}, {sd.month}, {sd.day})"

    for event in shifted.get("events", []):
        if event.get("name") == "Execute DAX Query":
            qt = event.get("metrics", {}).get("QueryText", "")
            if qt:
                qt = re.sub(START_P, lambda m: _shift(m, ">="), qt)
                qt = re.sub(END_P, lambda m: _shift(m, "<"), qt)
                event["metrics"]["QueryText"] = qt
    return shifted


def clear_aas_cache(model_name: str, workspace_name: str) -> tuple:
    """Clear AAS engine cache via XMLA."""
    try:
        datasets = fabric.list_datasets(workspace=workspace_name)
        row = datasets[datasets["Dataset Name"] == model_name]
        if len(row) == 0:
            return False, f"Dataset '{model_name}' not found"
        dataset_id = row["Dataset ID"].values[0]
        xmla_cmd = f"""
            <ClearCache xmlns="http://schemas.microsoft.com/analysisservices/2003/engine">
                <Object><DatabaseID>{dataset_id}</DatabaseID></Object>
            </ClearCache>"""
        fabric.execute_xmla(model_name, xmla_command=xmla_cmd, workspace=workspace_name)
        return True, "OK"
    except Exception as e:
        return False, str(e)[:80]


def refresh_fabric_dataset(model_name: str, workspace_name: str) -> tuple:
    """Trigger async dataset refresh."""
    try:
        fabric.refresh_dataset(dataset=model_name, workspace=workspace_name, refresh_type="full")
        return True, "OK"
    except Exception as e:
        return False, str(e)[:80]


def consolidate_iteration_csvs(base_log_path: str, loadtest_id: str, num_iterations: int) -> pl.DataFrame:
    """Load all iteration CSVs into a single Polars DataFrame with validation."""
    df_list = []
    missing_iters = []

    for iter_num in range(1, num_iterations + 1):
        iter_folder = f"{base_log_path}/{loadtest_id}_iter{iter_num}"
        try:
            csv_files = [f for f in os.listdir(iter_folder) if f.endswith(".csv")]
            for f in csv_files:
                df = pl.read_csv(os.path.join(iter_folder, f))
                if "iteration_number" not in df.columns:
                    df = df.with_columns(pl.lit(iter_num).alias("iteration_number"))
                df = df.with_columns([
                    pl.from_epoch(pl.col("start_time"), time_unit="s").dt.truncate("1s").dt.replace_time_zone("UTC").alias("start_time_s"),
                    pl.from_epoch(pl.col("start_time"), time_unit="s").dt.replace_time_zone("UTC").alias("start_time_dt")
                ])
                df_list.append(df)
        except FileNotFoundError:
            missing_iters.append(iter_num)

    if missing_iters:
        print(f"  WARNING: {len(missing_iters)} missing iteration folders (first: {missing_iters[0]})")

    if not df_list:
        return None
    return pl.concat(df_list)


def compute_all_stats(combined_df: pl.DataFrame) -> dict:
    """Compute overall, per-iteration, and per-session stats from one DataFrame."""
    min_start = combined_df.select(pl.col("start_time").min())[0, 0]
    max_end = combined_df.select((pl.col("start_time") + pl.col("duration")).max())[0, 0]

    df = combined_df.with_columns([
        ((pl.col("start_time") - min_start) * 1000).alias("offset_ms"),
        ((pl.col("start_time") - min_start) + pl.col("duration")).alias("completion_time")
    ])

    overall = df.select([
        pl.col("duration").min().alias("min"), pl.col("duration").max().alias("max"),
        pl.col("duration").mean().alias("avg"),
        pl.col("duration").quantile(0.50, interpolation="linear").alias("p50"),
        pl.col("duration").quantile(0.90, interpolation="linear").alias("p90"),
        pl.col("duration").quantile(0.95, interpolation="linear").alias("p95"),
        pl.col("duration").quantile(0.99, interpolation="linear").alias("p99"),
        pl.col("completion_time").max().alias("page_load"),
        pl.len().alias("count")
    ])

    per_iter = None
    if "iteration_number" in df.columns:
        per_iter = df.group_by("iteration_number").agg([
            pl.col("duration").mean().alias("avg_s"),
            pl.col("duration").quantile(0.50, interpolation="linear").alias("p50_s"),
            pl.col("duration").quantile(0.90, interpolation="linear").alias("p90_s"),
            pl.col("duration").quantile(0.95, interpolation="linear").alias("p95_s"),
            pl.col("duration").quantile(0.99, interpolation="linear").alias("p99_s"),
            ((pl.col("start_time") + pl.col("duration")).max() - pl.col("start_time").min()).alias("page_load_s"),
            pl.len().alias("query_count")
        ]).sort("iteration_number")
        per_iter = per_iter.with_columns([
            (pl.col("avg_s") * 1000).round(0).alias("avg_ms"),
            (pl.col("p50_s") * 1000).round(0).alias("p50_ms"),
            (pl.col("p90_s") * 1000).round(0).alias("p90_ms"),
            (pl.col("p95_s") * 1000).round(0).alias("p95_ms"),
            (pl.col("p99_s") * 1000).round(0).alias("p99_ms"),
            (pl.col("page_load_s") * 1000).round(0).alias("page_load_ms")
        ]).select(["iteration_number", "query_count", "avg_ms", "p50_ms", "p90_ms", "p95_ms", "p99_ms", "page_load_ms"])

    per_session = df.group_by("thread_id").agg([
        pl.col("duration").mean().alias("avg_ms"),
        pl.col("duration").quantile(0.50, interpolation="linear").alias("p50_ms"),
        pl.col("duration").quantile(0.90, interpolation="linear").alias("p90_ms"),
        pl.col("duration").quantile(0.95, interpolation="linear").alias("p95_ms"),
        pl.col("completion_time").max().alias("page_load_s"),
        pl.len().alias("query_count")
    ]).sort("thread_id")
    per_session = per_session.with_columns([
        (pl.col("avg_ms") * 1000).round(0).alias("avg_ms"),
        (pl.col("p50_ms") * 1000).round(0).alias("p50_ms"),
        (pl.col("p90_ms") * 1000).round(0).alias("p90_ms"),
        (pl.col("p95_ms") * 1000).round(0).alias("p95_ms"),
        (pl.col("page_load_s") * 1000).round(0).alias("page_load_ms")
    ]).select(["thread_id", "query_count", "avg_ms", "p50_ms", "p90_ms", "p95_ms", "page_load_ms"])

    return {
        "df": df, "overall": overall, "per_iter": per_iter, "per_session": per_session,
        "min_start": min_start, "max_end": max_end
    }

print("Helpers loaded")


In [ ]:
# Build Load Test Configuration

num_sessions = concurrent_users_dropdown.value
num_iterations = iterations_dropdown.value
parallel_queries = execution_mode_dropdown.value
max_parallel = max_parallel_dropdown.value
start_date = query_start_date.value
end_date = query_end_date.value
period_type = period_type_dropdown.value
workspace = environment_dropdown.value
test_label = test_label_input.value.strip()

if not test_label:
    raise ValueError("Test Label is required")
if end_date <= start_date:
    raise ValueError("End date must be after start date")

load_test_name = "Payments Performance Test"
dataset = "Payment Analytics Dataset"
query_folder = "/lakehouse/default/Files/test/PerfScenarios/Queries"
source_file = f"{query_folder}/PowerBIPerformanceData_Parameterized_New.json"
query_file = f"{query_folder}/PowerBIPerformanceData_Runtime.json"
iterations = 1

START_DATE_PATTERN = r">=\s*DATE\s*\(\s*\d{4}\s*,\s*\d{1,2}\s*,\s*\d{1,2}\s*\)"
END_DATE_PATTERN = r"<\s*DATE\s*\(\s*\d{4}\s*,\s*\d{1,2}\s*,\s*\d{1,2}\s*\)"
PERIOD_TYPE_PATTERN = r'TREATAS\s*\(\s*\{"(?:Daily|Weekly|Monthly|Yearly)"\}\s*,\s*\'DimRelativeDate\'\[Type\]\)'

new_start_date_str = f">= DATE({start_date.year}, {start_date.month}, {start_date.day})"
end_date_exclusive = end_date + td(days=1)
new_end_date_str = f"< DATE({end_date_exclusive.year}, {end_date_exclusive.month}, {end_date_exclusive.day})"
new_period_type_str = f'TREATAS({{"{period_type}"}}, \'DimRelativeDate\'[Type])'

source_content = notebookutils.fs.head("Files/test/PerfScenarios/Queries/PowerBIPerformanceData_Parameterized_New.json", 10000000)
data = json.loads(source_content)

# Filter out lightweight visuals (Date, Button, Slicer) - keep only perf-relevant queries
SKIP_VISUALS = {'Date', 'Button', 'Slicer', 'Period Type'}

events = data.get('events', [])
visual_map = {e['id']: e.get('metrics', {}).get('visualTitle', '')
              for e in events if e.get('name') == 'Visual Container Lifecycle'}
semantic_parent = {e['id']: e.get('parentId') for e in events if e.get('name') == 'Execute Semantic Query'}
query_parent = {e['id']: e.get('parentId') for e in events if e.get('name') == 'Query'}

def _get_visual_name(dax_event):
    sq_id = dax_event.get('parentId')
    q_id = semantic_parent.get(sq_id)
    v_id = query_parent.get(q_id)
    return visual_map.get(v_id, '')

skip_dax_ids = set()
for event in events:
    if event.get('name') == 'Execute DAX Query':
        vname = _get_visual_name(event)
        if any(skip in vname for skip in SKIP_VISUALS):
            skip_dax_ids.add(event.get('id'))

data['events'] = [e for e in events if e.get('id') not in skip_dax_ids]

# Parameterize remaining queries
total_queries = 0
for event in data.get('events', []):
    if event.get('name') == 'Execute DAX Query':
        total_queries += 1
        qt = event.get('metrics', {}).get('QueryText', '')
        if qt:
            qt = re.sub(START_DATE_PATTERN, new_start_date_str, qt)
            qt = re.sub(END_DATE_PATTERN, new_end_date_str, qt)
            qt = re.sub(PERIOD_TYPE_PATTERN, new_period_type_str, qt)
            event['metrics']['QueryText'] = qt

print(f"  Filtered: {len(skip_dax_ids)} skipped, {total_queries} kept")

notebookutils.fs.put("Files/test/PerfScenarios/Queries/PowerBIPerformanceData_Runtime.json", json.dumps(data, indent=2), True)

ts = time.strftime("%Y%m%d-%H%M%S")
loadtestId = f"{load_test_name}-{ts}"
notebookutils.fs.mkdirs(f"Files/test/PerfScenarios/logs/{loadtestId}")

base_args = {
    "xmla_endpoint": f"powerbi://api.powerbi.com/v1.0/myorg/{workspace}",
    "model": dataset, "roles": None, "customdata": None, "effective_username": None,
    "iterations": iterations, "delay_sec": 1, "loadtestId": loadtestId,
    "concurrent_threads": num_sessions, "useRootDefaultLakehouse": True,
    "parallel_queries": parallel_queries, "max_parallel_queries": max_parallel,
    "perf_analyzer_filename": query_file
}

print(f"Load Test: {loadtestId}")
print(f"  Label: {test_label} | Workspace: {workspace}")
print(f"  Sessions: {num_sessions} | Iterations: {num_iterations} | Queries/iter: {total_queries}")
print(f"  Dates: {start_date} to {end_date} | Period: {period_type}")


In [ ]:
# Execute Load Test (Minimal Output for Long Runs)

clear_cache_per_iter = clear_aas_cache_checkbox.value
refresh_fabric_per_iter = refresh_fabric_checkbox.value
session_start_time = datetime.now()
session_log_folder = f"/lakehouse/default/Files/test/PerfScenarios/logs/{loadtestId}"
iteration_records = []
failed_iterations = 0

print(f"SESSION: {loadtestId} | {num_iterations} iterations | Started: {session_start_time.strftime('%H:%M:%S')}")

for iter_num in range(1, num_iterations + 1):
    day_offset = iter_num - 1
    iter_loadtestId = f"{loadtestId}_iter{iter_num}"
    iter_start_time = datetime.now()

    if refresh_fabric_per_iter:
        refresh_fabric_dataset(dataset, workspace)
    if clear_cache_per_iter:
        clear_aas_cache(dataset, workspace)

    iter_runtime = shift_dates_in_runtime_json(data, day_offset)
    iter_runtime_filename = f"runtime_{loadtestId}_iter{iter_num}.json"
    notebookutils.fs.put(f"Files/test/PerfScenarios/RunTime/{iter_runtime_filename}", json.dumps(iter_runtime, indent=2), True)
    iter_runtime_path = f"/lakehouse/default/Files/test/PerfScenarios/RunTime/{iter_runtime_filename}"

    DAG = {"activities": [], "concurrency": num_sessions}
    for thread_id in range(1, num_sessions + 1):
        thread_args = base_args.copy()
        thread_args.update({
            "threadId": thread_id, "loadtestId": iter_loadtestId,
            "iterations": 1, "perf_analyzer_filename": iter_runtime_path,
            "iteration_number": iter_num, "date_offset_days": day_offset
        })
        DAG["activities"].append({
            "name": f"iter{iter_num}_user{thread_id}",
            "path": "RunPerfScenario_Parallel",
            "args": thread_args,
            "timeoutPerCellInSeconds": 300
        })

    dag_start = time.time()
    try:
        notebookutils.notebook.runMultiple(DAG, {"displayDAGVia498WorkAround": True})
        dag_duration = time.time() - dag_start
        dag_status = "SUCCESS"
    except Exception as e:
        dag_duration = time.time() - dag_start
        dag_status = f"FAILED: {str(e)[:50]}"
        failed_iterations += 1

    iter_end_time = datetime.now()
    iter_duration = (iter_end_time - iter_start_time).total_seconds()

    iteration_records.append({
        "iteration": iter_num, "start_time": iter_start_time.isoformat(),
        "end_time": iter_end_time.isoformat(), "duration_sec": round(iter_duration, 2),
        "date_offset_days": day_offset, "dag_status": dag_status,
        "dag_duration_sec": round(dag_duration, 2),
        "fabric_refresh": refresh_fabric_per_iter, "cache_cleared": clear_cache_per_iter
    })

    # Compact progress: every 10 iterations or on failure
    if iter_num % 10 == 0 or iter_num == num_iterations or "FAILED" in dag_status:
        elapsed = (datetime.now() - session_start_time).total_seconds()
        avg_iter = elapsed / iter_num
        eta = avg_iter * (num_iterations - iter_num)
        status = f" [{dag_status}]" if "FAILED" in dag_status else ""
        print(f"  [{iter_num}/{num_iterations}] {iter_num/num_iterations*100:.0f}% | elapsed {elapsed:.0f}s | ETA {eta:.0f}s | failed: {failed_iterations}{status}")

    if iter_num < num_iterations:
        time.sleep(3)

session_end_time = datetime.now()
session_duration = (session_end_time - session_start_time).total_seconds()
print(f"\nDONE: {session_duration:.0f}s total | {failed_iterations} failed")


In [ ]:
# Consolidate, Analyze & Export Results

base_log_path = "/lakehouse/default/Files/test/PerfScenarios/logs"
combined_df = consolidate_iteration_csvs(base_log_path, loadtestId, num_iterations)

if combined_df is None:
    print("ERROR: No CSV files found")
else:
    iter_count = combined_df["iteration_number"].n_unique()
    print(f"Consolidated: {len(combined_df)} records | {iter_count} iterations")

    stats = compute_all_stats(combined_df)
    overall_stats = stats["overall"]
    combined_df = stats["df"]
    session_stats_df = stats["per_session"]

    start_utc = dt.fromtimestamp(stats["min_start"], tz=timezone.utc)
    end_utc = dt.fromtimestamp(stats["max_end"], tz=timezone.utc)
    pst = timezone(td(hours=-8))
    start_pst = start_utc.astimezone(pst)
    end_pst = end_utc.astimezone(pst)
    total_duration_s = stats["max_end"] - stats["min_start"]

    print(f"Stats: avg={overall_stats['avg'][0]*1000:.0f}ms, p50={overall_stats['p50'][0]*1000:.0f}ms, "
          f"p90={overall_stats['p90'][0]*1000:.0f}ms, p95={overall_stats['p95'][0]*1000:.0f}ms, "
          f"p99={overall_stats['p99'][0]*1000:.0f}ms")

    if VERBOSE and stats["per_iter"] is not None:
        print("\nPer-Iteration Stats:")
        print(stats["per_iter"])

    # ===== Export =====
    end_date_str = datetime.now().strftime("%Y-%m-%d")
    results_folder = f"/lakehouse/default/Files/test/PerfScenarios/results/{end_date_str}/{loadtestId}"
    results_folder_relative = f"Files/test/PerfScenarios/results/{end_date_str}/{loadtestId}"
    notebookutils.fs.mkdirs(results_folder_relative)

    # Raw results
    results_pdf = combined_df.to_pandas()
    results_pdf["test_label"] = test_label
    results_pdf.to_csv(f"{results_folder}/results.csv", index=False)

    # Summary stats
    query_summary_stats = combined_df.group_by("query_number", "visual_name").agg([
        pl.col("duration").min().alias("min_s"), pl.col("duration").max().alias("max_s"),
        pl.col("duration").mean().alias("avg_s"), pl.col("duration").quantile(0.50).alias("p50_s"),
        pl.col("duration").quantile(0.90).alias("p90_s"), pl.col("duration").quantile(0.95).alias("p95_s"),
        pl.col("duration").quantile(0.99).alias("p99_s"), pl.len().alias("execution_count")
    ]).sort("query_number")

    summary_rows = [{
        "test_label": test_label, "query_number": r["query_number"], "visual": r["visual_name"],
        "start_date": str(start_date), "end_date": str(end_date), "period_type": period_type,
        "min_ms": round(r["min_s"]*1000, 2), "max_ms": round(r["max_s"]*1000, 2),
        "avg_ms": round(r["avg_s"]*1000, 2), "p50_ms": round(r["p50_s"]*1000, 2),
        "p90_ms": round(r["p90_s"]*1000, 2), "p95_ms": round(r["p95_s"]*1000, 2),
        "p99_ms": round(r["p99_s"]*1000, 2), "execution_count": r["execution_count"]
    } for r in query_summary_stats.iter_rows(named=True)]

    summary_rows.append({
        "test_label": test_label, "query_number": "ALL", "visual": "OVERALL",
        "start_date": str(start_date), "end_date": str(end_date), "period_type": period_type,
        "min_ms": round(overall_stats["min"][0]*1000, 2), "max_ms": round(overall_stats["max"][0]*1000, 2),
        "avg_ms": round(overall_stats["avg"][0]*1000, 2), "p50_ms": round(overall_stats["p50"][0]*1000, 2),
        "p90_ms": round(overall_stats["p90"][0]*1000, 2), "p95_ms": round(overall_stats["p95"][0]*1000, 2),
        "p99_ms": round(overall_stats["p99"][0]*1000, 2),
        "execution_count": int(overall_stats["count"][0])
    })
    pd.DataFrame(summary_rows).to_csv(f"{results_folder}/summary_stats.csv", index=False)

    # Session telemetry JSON
    telemetry = {
        "session_id": loadtestId, "test_label": test_label,
        "session_start_time": session_start_time.isoformat(),
        "session_end_time": session_end_time.isoformat(),
        "session_duration_sec": round(session_duration, 2),
        "concurrent_users": num_sessions, "num_iterations": num_iterations,
        "failed_iterations": failed_iterations,
        "iterations": iteration_records
    }
    notebookutils.fs.put(f"Files/test/PerfScenarios/logs/{loadtestId}/session_telemetry.json", json.dumps(telemetry, indent=2), True)

    # Metadata JSON
    notebookutils.fs.put(f"{results_folder_relative}/metadata.json", json.dumps({
        "test_label": test_label, "loadtest_id": loadtestId, "model": dataset,
        "workspace": workspace, "concurrent_threads": num_sessions,
        "iterations": num_iterations, "total_queries": int(overall_stats["count"][0]),
        "query_parameters": {"start_date": str(start_date), "end_date": str(end_date), "period_type": period_type},
        "test_date": end_date_str
    }, indent=2, default=str), True)

    print(f"Exported to: {results_folder}")
    print(f"  results.csv | summary_stats.csv ({len(summary_rows)} rows) | metadata.json")


In [ ]:
# Send Telemetry to App Insights

if combined_df is not None:
    per_iteration_stats = []
    if "iteration_number" in combined_df.columns:
        for iter_num in sorted(combined_df["iteration_number"].unique().to_list()):
            iter_df = combined_df.filter(pl.col("iteration_number") == iter_num)
            if len(iter_df) > 0:
                s = iter_df.select([
                    pl.col("duration").mean().alias("avg"),
                    pl.col("duration").quantile(0.50).alias("p50"),
                    pl.col("duration").quantile(0.90).alias("p90"),
                    pl.col("duration").quantile(0.95).alias("p95"),
                    pl.col("duration").quantile(0.99).alias("p99"),
                    pl.len().alias("count")
                ])
                per_iteration_stats.append({
                    "iteration": iter_num,
                    "query_count": int(s["count"][0]),
                    "avg_ms": round(s["avg"][0] * 1000, 2),
                    "p50_ms": round(s["p50"][0] * 1000, 2),
                    "p90_ms": round(s["p90"][0] * 1000, 2),
                    "p95_ms": round(s["p95"][0] * 1000, 2),
                    "p99_ms": round(s["p99"][0] * 1000, 2)
                })

    try:
        result = send_loadtest_telemetry(
            test_label=test_label, loadtest_id=loadtestId,
            dataset=dataset, workspace=workspace,
            num_sessions=num_sessions, iterations=iterations,
            parallel_queries=parallel_queries, max_parallel=max_parallel,
            start_date=start_date, end_date=end_date, period_type=period_type,
            overall_stats=overall_stats, combined_df=combined_df,
            summary_rows=summary_rows, session_stats=session_stats_df,
            start_utc=start_utc, end_utc=end_utc,
            start_pst=start_pst, end_pst=end_pst,
            total_duration_s=total_duration_s, environment="test",
            iteration_records=iteration_records,
            per_iteration_stats=per_iteration_stats if per_iteration_stats else None,
            num_iterations=num_iterations
        )
        print(f"Telemetry sent: {result['events_sent']} events, {result['metrics_sent']} metrics")
        print(f"Test Run ID: {result['test_run_id']}")
    except Exception as e:
        print(f"Telemetry FAILED (results still saved): {e}")
else:
    print("No data to send")
